# SEC Filing Financial Analyzer

**Author:** (your name here) · **Type:** Quantitative Finance / Equity Research Automation
**Stack:** Python, SEC EDGAR XBRL API, pandas, matplotlib, seaborn

---

## 1. Project Overview

This project automates what equity analysts otherwise do by hand: opening a
company's 10-K / 10-Q filings, finding the income statement and balance
sheet, and computing financial ratios. Instead of scraping PDFs (fragile and
inconsistent), this pulls **structured, machine-readable XBRL data directly
from the SEC's free EDGAR API** — every public company has been required to
tag its filings this way since ~2009, which means the same revenue or
net-income figure can be retrieved the same way for Apple as for any other
filer.

The pipeline:
1. Resolve ticker symbols to SEC CIK numbers.
2. Pull each company's full XBRL "company facts" history from EDGAR.
3. Extract standardized line items (revenue, net income, assets, equity,
   etc.), handling the fact that different companies tag the same concept
   under slightly different XBRL tags.
4. Clean and deduplicate multi-year filing history into one tidy table per
   company.
5. Compute standard financial ratios (margins, ROE, ROA, leverage, liquidity).
6. Compare multiple companies side-by-side, and each company against its own
   history over time.
7. Visualize trends and export clean CSVs.

## 2. Real-World Finance Use Case

Equity research analysts and credit analysts spend a meaningful chunk of
their week doing exactly this extraction by hand for every company they
cover, every quarter — pulling the same handful of numbers out of a new
filing and recomputing the same ratios. This project automates that
extraction layer entirely, using the SEC's own structured data feed (the same
one Bloomberg, FactSet, and other terminal vendors are themselves built on
top of). It's the unglamorous data-plumbing work that sits underneath every
fundamental analysis dashboard you've ever seen.

## 3. System Architecture

```
 ┌────────────────┐     ┌──────────────────┐     ┌────────────────────┐
 │  Ticker → CIK   │ --> │  EDGAR XBRL Pull  │ --> │  Concept Extraction│
 │   resolution    │     │ (companyfacts API)│     │ (tag fallback logic)│
 └────────────────┘     └──────────────────┘     └────────────────────┘
                                                           │
                                                           v
                                                ┌────────────────────┐
                                                │  Ratio Engine       │
                                                │ (margins, ROE, ROA, │
                                                │  leverage, liquidity)│
                                                └────────────────────┘
                                                           │
                                                           v
                                                ┌────────────────────┐
                                                │  Reporting Layer    │
                                                │ (charts, comparison │
                                                │  tables, CSV export)│
                                                └────────────────────┘
```

## 4. Required APIs and Data Sources
- **SEC EDGAR XBRL API** (`data.sec.gov/api/xbrl/companyfacts/...`) — free, no
  API key, but **requires a custom `User-Agent` header identifying you**
  (name + email). SEC will return `403 Forbidden` without one — this is a
  fair-access policy, not a bug. **You must edit this in the config cell
  below before running.**
- **SEC company tickers file** (`sec.gov/files/company_tickers.json`) — free
  lookup table mapping ticker symbols to CIK numbers.

## 5. Required Python Libraries
`requests`, `pandas`, `numpy`, `matplotlib`, `seaborn`

## 6. Folder/File Structure (conceptual)
```
sec-filing-analyzer/
├── README.md
├── sec_filing_analyzer.ipynb     <- this notebook
├── /outputs
│    ├── financials_raw.csv
│    ├── ratios_comparison.csv
│    ├── revenue_trend.png
│    ├── margin_trend.png
│    └── ratio_comparison.png
└── requirements.txt
```


## 7. Step 0 — Install & import dependencies

In [ ]:
!pip install requests pandas numpy matplotlib seaborn --quiet --upgrade
print("Dependencies ready.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

os.makedirs("/content/outputs", exist_ok=True)
print("Imports ready.")

## Step 1 — Configuration

**IMPORTANT:** SEC EDGAR will block requests (`403 Forbidden`) unless you
identify yourself with a real name + email in the User-Agent header. This is
SEC's actual fair-access policy — it is not optional. Edit the line below
before running anything else.

In [ ]:
CONFIG = {
    # >>> REQUIRED: replace with your real name + a real email. SEC blocks
    # generic/missing User-Agents. Format SEC asks for: "Sample Company Name AdminContact@domain.com"
    "user_agent": "YourName YourEmail@example.com",

    "tickers": ["AAPL", "MSFT", "GOOGL"],   # companies to compare
    "form_types": ["10-K"],                  # "10-K" = annual, "10-Q" = quarterly
    "years_back": 6,

    # Hardcoded fallback CIKs in case the ticker-lookup file fetch fails
    # (network hiccup, rate limiting). Add more here if you expand "tickers".
    "fallback_ciks": {
        "AAPL": 320193, "MSFT": 789019, "GOOGL": 1652044,
        "AMZN": 1018724, "NVDA": 1045810, "META": 1326801,
        "TSLA": 1318605, "JPM": 19617,
    },

    # SEC asks for no more than ~10 requests/second; we go well under that.
    "request_delay_sec": 0.3,
}

HEADERS = {"User-Agent": CONFIG["user_agent"]}

if "example.com" in CONFIG["user_agent"]:
    print("WARNING: you have not set a real User-Agent yet — SEC will likely")
    print("reject requests with 403 Forbidden until you edit CONFIG[\"user_agent\"].")

## 8. Step 2 — Data Collection Pipeline

### 2a. Ticker → CIK resolution
SEC identifies companies by **CIK** (Central Index Key), not ticker symbol.
We fetch the official ticker→CIK mapping file, with a hardcoded fallback in
case that fetch is blocked or rate-limited.

In [ ]:
def get_cik_lookup(headers, fallback_ciks):
    """Fetch ticker -> CIK mapping from SEC. Falls back to a hardcoded dict
    for known tickers if the live fetch fails, rather than crashing."""
    url = "https://www.sec.gov/files/company_tickers.json"
    try:
        resp = requests.get(url, headers=headers, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        lookup = {row["ticker"].upper(): row["cik_str"] for row in data.values()}
        print(f"Loaded live CIK lookup: {len(lookup)} tickers.")
        return lookup
    except Exception as e:
        print(f"WARNING: live CIK lookup failed ({e}). Using hardcoded fallback list.")
        return dict(fallback_ciks)


def resolve_cik(ticker, lookup, fallback_ciks):
    ticker = ticker.upper()
    if ticker in lookup:
        return int(lookup[ticker])
    if ticker in fallback_ciks:
        return int(fallback_ciks[ticker])
    raise ValueError(f"Could not resolve CIK for ticker '{ticker}'. Add it to fallback_ciks in CONFIG.")


cik_lookup = get_cik_lookup(HEADERS, CONFIG["fallback_ciks"])

ticker_to_cik = {}
for t in CONFIG["tickers"]:
    try:
        ticker_to_cik[t] = resolve_cik(t, cik_lookup, CONFIG["fallback_ciks"])
    except ValueError as e:
        print(f"WARNING: {e} -- skipping {t}.")

print("\nResolved CIKs:")
for t, c in ticker_to_cik.items():
    print(f"  {t}: CIK {c:010d}")

### 2b. Pull XBRL company facts for each ticker
Each call hits `data.sec.gov` once per company. We rate-limit politely and handle the common failure modes explicitly (403 = bad User-Agent, 404 = bad CIK, timeouts, malformed JSON).

In [ ]:
def get_company_facts(cik, headers, delay=0.3):
    """Pull the full XBRL company-facts payload for a given CIK."""
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json"
    time.sleep(delay)
    try:
        resp = requests.get(url, headers=headers, timeout=20)
    except requests.RequestException as e:
        raise RuntimeError(f"Network error fetching CIK {cik}: {e}")

    if resp.status_code == 403:
        raise RuntimeError(
            "403 Forbidden -- SEC rejected the request. This almost always means "
            "CONFIG['user_agent'] is missing/invalid. Set it to a real name + email."
        )
    if resp.status_code == 404:
        raise RuntimeError(f"404 Not Found -- no XBRL facts for CIK {cik} (wrong CIK?).")
    resp.raise_for_status()

    try:
        return resp.json()
    except json.JSONDecodeError as e:
        raise RuntimeError(f"Malformed JSON response for CIK {cik}: {e}")


company_facts = {}
for t, cik in ticker_to_cik.items():
    try:
        company_facts[t] = get_company_facts(cik, HEADERS, CONFIG["request_delay_sec"])
        print(f"Fetched company facts for {t} (CIK {cik}).")
    except RuntimeError as e:
        print(f"WARNING: skipping {t} -- {e}")

print(f"\nSuccessfully pulled data for {len(company_facts)}/{len(ticker_to_cik)} companies.")

## 9. Step 3 — Concept Extraction (Cleaning & Feature Engineering)

Different companies tag economically identical concepts under different XBRL
tags (e.g. some use `Revenues`, others `RevenueFromContractWithCustomerExcludingAssessedTax`).
We define a **fallback tag list per concept** and take the first one that
exists for each company. We also deduplicate: the same fiscal year's figure
often appears in multiple filings (as a comparative prior-year number), so we
keep only the most recently filed value for each period.

In [ ]:
# Fallback tag lists -- ordered by how common each tag is across filers.
CONCEPT_TAGS = {
    "Revenue":            ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax", "SalesRevenueNet"],
    "CostOfRevenue":      ["CostOfRevenue", "CostOfGoodsAndServicesSold", "CostOfGoodsSold"],
    "GrossProfit":        ["GrossProfit"],
    "OperatingIncome":    ["OperatingIncomeLoss"],
    "NetIncome":          ["NetIncomeLoss", "ProfitLoss"],
    "TotalAssets":        ["Assets"],
    "TotalLiabilities":   ["Liabilities"],
    "StockholdersEquity": ["StockholdersEquity", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"],
    "CurrentAssets":      ["AssetsCurrent"],
    "CurrentLiabilities": ["LiabilitiesCurrent"],
    "CashAndEquivalents": ["CashAndCashEquivalentsAtCarryingValue"],
    "LongTermDebt":       ["LongTermDebtNoncurrent", "LongTermDebt"],
    "EPS_Diluted":        ["EarningsPerShareDiluted"],
}


def extract_concept_raw(facts_json, tag_candidates, unit="USD"):
    """Return (DataFrame of raw XBRL records, tag_used) for the first tag in
    tag_candidates that exists with data, or (None, None) if none do."""
    us_gaap = facts_json.get("facts", {}).get("us-gaap", {})
    for tag in tag_candidates:
        node = us_gaap.get(tag)
        if node and unit in node.get("units", {}):
            records = node["units"][unit]
            if records:
                return pd.DataFrame(records), tag
    return None, None


def annualize(df_raw, form_types):
    """Collapse raw XBRL records into one clean value per fiscal year:
    keep only the requested form types, dedupe by period end date (keeping
    the most recently FILED value), and index by fiscal year."""
    if df_raw is None or df_raw.empty:
        return pd.Series(dtype=float)

    df = df_raw[df_raw["form"].isin(form_types)].copy()
    if df.empty:
        df = df_raw.copy()  # fallback: use whatever forms are available

    df["filed"] = pd.to_datetime(df["filed"])
    df["end"] = pd.to_datetime(df["end"])
    df = df.sort_values("filed").drop_duplicates(subset=["end"], keep="last")
    df["fiscal_year"] = df["end"].dt.year

    series = df.set_index("fiscal_year")["val"].sort_index()
    series = series[~series.index.duplicated(keep="last")]
    return series


def build_financials_table(facts_json, concept_tags, form_types, years_back):
    """Build one tidy DataFrame (rows = fiscal year, columns = line items)
    for a single company."""
    cols = {}
    tags_used = {}
    for concept, candidates in concept_tags.items():
        raw, tag = extract_concept_raw(facts_json, candidates)
        series = annualize(raw, form_types)
        cols[concept] = series
        tags_used[concept] = tag

    table = pd.DataFrame(cols)
    cutoff_year = datetime.today().year - years_back
    table = table[table.index >= cutoff_year]
    return table.sort_index(), tags_used


financials = {}
tag_log = {}
for t, facts in company_facts.items():
    table, tags_used = build_financials_table(facts, CONCEPT_TAGS, CONFIG["form_types"], CONFIG["years_back"])
    financials[t] = table
    tag_log[t] = tags_used
    missing = [c for c, tag in tags_used.items() if tag is None]
    if missing:
        print(f"{t}: no data found for {missing} (left as NaN, ratios using these will be NaN).")

print("\nExample -- extracted financials table:")
financials[CONFIG["tickers"][0]]

## 10. Step 4 — Ratio Engine (Core Algorithms)

Standard fundamental-analysis ratios, computed directly from the cleaned
financials table. Every ratio is guarded against division-by-zero / missing
data, returning `NaN` rather than crashing, so one missing line item doesn't
take down the whole pipeline.

In [ ]:
def safe_div(a, b):
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where((b == 0) | pd.isna(b) | pd.isna(a), np.nan, a / b)


def compute_ratios(table):
    """Compute standard ratios from a financials table. Returns a new
    DataFrame indexed the same way (by fiscal year)."""
    r = pd.DataFrame(index=table.index)

    gross_profit = table.get("GrossProfit")
    if gross_profit is None or gross_profit.isna().all():
        gross_profit = table.get("Revenue") - table.get("CostOfRevenue")

    r["Gross Margin"]      = safe_div(gross_profit.values, table.get("Revenue").values)
    r["Operating Margin"]  = safe_div(table.get("OperatingIncome").values, table.get("Revenue").values)
    r["Net Margin"]        = safe_div(table.get("NetIncome").values, table.get("Revenue").values)
    r["ROE"]               = safe_div(table.get("NetIncome").values, table.get("StockholdersEquity").values)
    r["ROA"]               = safe_div(table.get("NetIncome").values, table.get("TotalAssets").values)
    r["Debt-to-Equity"]    = safe_div(table.get("TotalLiabilities").values, table.get("StockholdersEquity").values)
    r["Current Ratio"]     = safe_div(table.get("CurrentAssets").values, table.get("CurrentLiabilities").values)
    r["Revenue YoY Growth"] = table.get("Revenue").pct_change()
    r["Net Income YoY Growth"] = table.get("NetIncome").pct_change()
    return r


ratios = {t: compute_ratios(tbl) for t, tbl in financials.items() if not tbl.empty}

print("Example -- computed ratios:")
ratios[CONFIG["tickers"][0]].map(lambda x: f"{x:.2%}" if pd.notna(x) else "n/a")

## 11. Step 5 — Cross-Company Comparison
Build a single comparison table using each company's most recent available fiscal year.

In [ ]:
latest_rows = []
for t, r in ratios.items():
    if r.empty:
        continue
    latest_year = r.index.max()
    row = r.loc[latest_year].copy()
    row.name = t
    row["Fiscal Year"] = latest_year
    latest_rows.append(row)

comparison_table = pd.DataFrame(latest_rows)
cols_order = ["Fiscal Year", "Gross Margin", "Operating Margin", "Net Margin",
              "ROE", "ROA", "Debt-to-Equity", "Current Ratio",
              "Revenue YoY Growth", "Net Income YoY Growth"]
comparison_table = comparison_table[[c for c in cols_order if c in comparison_table.columns]]

display_table = comparison_table.copy()
for c in display_table.columns:
    if c != "Fiscal Year":
        display_table[c] = display_table[c].map(lambda x: f"{x:.1%}" if pd.notna(x) else "n/a")

display_table

## Visualizations & Dashboard Components

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for t, tbl in financials.items():
    if "Revenue" in tbl and not tbl["Revenue"].isna().all():
        axes[0].plot(tbl.index, tbl["Revenue"] / 1e9, marker="o", label=t)
    if "NetIncome" in tbl and not tbl["NetIncome"].isna().all():
        axes[1].plot(tbl.index, tbl["NetIncome"] / 1e9, marker="o", label=t)

axes[0].set_title("Revenue Trend ($B)", fontweight="bold")
axes[1].set_title("Net Income Trend ($B)", fontweight="bold")
for ax in axes:
    ax.set_xlabel("Fiscal Year")
    ax.legend()
plt.tight_layout()
plt.savefig("/content/outputs/revenue_trend.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for t, r in ratios.items():
    if "Net Margin" in r and not r["Net Margin"].isna().all():
        ax.plot(r.index, r["Net Margin"], marker="o", label=f"{t} Net Margin")

ax.set_title("Net Margin Trend by Company", fontsize=13, fontweight="bold")
ax.set_xlabel("Fiscal Year")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend()
plt.tight_layout()
plt.savefig("/content/outputs/margin_trend.png", dpi=150)
plt.show()

In [ ]:
ratio_cols = ["Gross Margin", "Operating Margin", "Net Margin", "ROE", "ROA"]
plot_df = comparison_table[ratio_cols].astype(float)

fig, ax = plt.subplots(figsize=(11, 5.5))
plot_df.plot(kind="bar", ax=ax, width=0.75)
ax.set_title("Latest-Fiscal-Year Ratio Comparison", fontsize=13, fontweight="bold")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend(title="Ratio", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("/content/outputs/ratio_comparison.png", dpi=150)
plt.show()

## 12. Data Coverage / Quality Metrics

Since this pulls real, occasionally inconsistent filer data, it's worth
reporting **how complete** the extraction actually was -- which line items
were found vs. missing for each company, so you know which ratios to trust.

In [ ]:
coverage_rows = []
for t, tags_used in tag_log.items():
    found = sum(1 for v in tags_used.values() if v is not None)
    total = len(tags_used)
    coverage_rows.append({"Ticker": t, "Concepts Found": found, "Concepts Total": total,
                           "Coverage": found / total})

coverage_df = pd.DataFrame(coverage_rows).set_index("Ticker")
coverage_df["Coverage"] = coverage_df["Coverage"].map(lambda x: f"{x:.0%}")
coverage_df

## 13. Final Deliverables — Export Results

In [ ]:
for t, tbl in financials.items():
    tbl.to_csv(f"/content/outputs/{t}_financials_raw.csv")
    if t in ratios:
        ratios[t].to_csv(f"/content/outputs/{t}_ratios.csv")

comparison_table.to_csv("/content/outputs/ratios_comparison_latest.csv")
coverage_df.to_csv("/content/outputs/data_coverage.csv")

print("Saved to /content/outputs/:")
for f in sorted(os.listdir("/content/outputs")):
    print(" -", f)

## 14. Resume Description

> **SEC Filing Financial Analyzer** — Built an automated equity-research tool
> in Python that pulls structured XBRL financial data directly from the SEC
> EDGAR API for any public company, handling ticker-to-CIK resolution, tag
> fallback logic across inconsistently-tagged filers, deduplication of
> restated/comparative filing data, and computation of standard fundamental
> ratios (margins, ROE, ROA, leverage, liquidity, YoY growth). Built
> cross-company comparison views and multi-year trend visualizations, with
> defensive error handling for SEC's rate limits and access-control
> requirements.

## 15. Potential Upgrades
- Add **10-Q (quarterly)** support alongside 10-K for more granular trend lines.
- Pull the **MD&A and Risk Factors text sections** from filings and run NLP
  sentiment / topic analysis on them (natural next phase of this project).
- Add **peer-group benchmarking** — automatically pull an entire sector/index
  and rank companies by each ratio.
- Cache `company_facts` responses to disk so reruns don't re-hit the API.
- Add valuation ratios (P/E, EV/EBITDA) by merging in live price data (e.g.
  via the `yfinance`-based pipeline from the Black-Litterman project).
- Wrap in a Streamlit app where you type a ticker and get a live one-page
  fundamental summary.
